In [0]:
from multiprocessing.pool import ThreadPool
import os
import hashlib


In [0]:
print(os.listdir("/Volumes/Projeto_Engenharia/bronze/arquivos_raw"))

## 🏗️ Estratégia de Ingestão - Camada Bronze

**Objetivo:** Converter arquivos brutos para tabelas Delta de forma eficiente e incremental.

* [ ] **Carga Incremental:** Inserir apenas arquivos novos ou que sofreram alterações.
* [ ] **Processamento em Lote (Batching):** Processar arquivos em grupos (ex: 4 por vez) para controle de recursos.
* [ ] **Fidelidade à Origem:** Nenhuma transformação de negócio; apenas conversão de CSV para Delta Lake.

In [0]:
#1 - Configurações (aqui você controla tudo)
CATALOG = "Projeto_Engenharia"
SCHEMA_BRONZE = "bronze"
VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA_BRONZE}/arquivos_raw"
BATCH_SIZE = 4  # máximo de arquivos por vez

# Arquivos que você quer IGNORAR (tira daqui se não quiser subir)
BLACKLIST = [
    
]

# 2 - Listar e filtrar arquivos
todos_arquivos = os.listdir(VOLUME_PATH)
tabelas = [f.split('_202')[0] for f in todos_arquivos]

# Remove os da blacklist
tabelas_filtradas = [t for t in tabelas if t not in BLACKLIST]

print(f"Total encontrado: {len(tabelas)}")
print(f"Ignorados: {BLACKLIST}")
print(f"Para processar: {tabelas_filtradas}")

# 3 - Função que verifica se o arquivo mudou (hash)
def get_file_hash(path):
    """Gera hash do arquivo pra saber se mudou"""
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def arquivo_mudou(tabela, arquivo_path):
    """Compara hash atual com o salvo — True se mudou"""
    hash_table = f"{CATALOG}.{SCHEMA_BRONZE}.file_control"
    hash_atual = get_file_hash(arquivo_path)
    
    try:
        df_control = spark.sql(f"""
            SELECT file_hash FROM {hash_table}
            WHERE table_name = '{tabela}'
        """)
        if df_control.count() == 0:
            return True  # nunca foi carregado
        
        hash_salvo = df_control.collect()[0][0]
        return hash_atual != hash_salvo  # True se mudou
    except:
        return True  # tabela de controle não existe ainda

#  4 - Tabela de controle (roda uma vez)
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA_BRONZE}.file_control (
        table_name STRING,
        file_hash STRING,
        loaded_at TIMESTAMP
    )
""")

# 5 - Função principal de ingestão
def processar_arquivo(tabela):
    arquivo=None
    for i in todos_arquivos:
        if i.startswith(tabela):
            arquivo=i
            break

    arquivo_path = f"{VOLUME_PATH}/{arquivo}"
    
    if not arquivo:
        print(f"❌ {tabela} — arquivo não encontrado, pulando...")
        return

    if not arquivo_mudou(tabela, arquivo_path):
        print(f"⏭️  {tabela} — sem alterações, pulando...")
        return
    
    print(f"⚙️  Processando {tabela}...")
    
    df = spark.read.format("csv") \
        .option("header", "true") \
        .option("inferSchema", "false")\
        .load(arquivo_path)
    
    # Metadados de rastreabilidade
    df = df.withColumn("_source_file", F.lit(tabela)) \
           .withColumn("_loaded_at", F.current_timestamp())
    
    # Salva como Delta raw
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"{CATALOG}.{SCHEMA_BRONZE}.{tabela.lower()}")


# 8 - Processa em batches de 4 

for i in range(0, len(tabelas_filtradas), BATCH_SIZE):
    batch = tabelas_filtradas[i:i+BATCH_SIZE] # cada vez que rodar pegar o indice e pegar os proximos 4
    print(f"\n📦 Batch {i//BATCH_SIZE + 1}: {batch}")
    
    with ThreadPool(BATCH_SIZE) as pool:
        pool.map(processar_arquivo, batch)

print("\n🎉 Ingestão finalizada!")

In [0]:
for i in range(0, len(tabelas_filtradas), BATCH_SIZE):
    print(tabelas_filtradas[i:i+BATCH_SIZE])